In [2]:
import csv
f = open("set_covering.csv")
csvfile = csv.DictReader(f, delimiter='\t')

headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()

In [3]:
len(table)

17

In [4]:
table

[{'District': 'Cost',
  'D1': '600',
  'D2': '680',
  'D3': '560',
  'D4': '880',
  'D5': '670',
  'D6': '740',
  'D7': '590',
  'D8': '900',
  'D9': '920',
  'D10': '460',
  'D11': '910',
  'D12': '800',
  'D13': '720',
  'D14': '500',
  'D15': '640',
  'D16': '400'},
 {'District': 'D1',
  'D1': '1',
  'D2': '1',
  'D3': '',
  'D4': '1',
  'D5': '1',
  'D6': '',
  'D7': '',
  'D8': '',
  'D9': '',
  'D10': '',
  'D11': '',
  'D12': '',
  'D13': '',
  'D14': '',
  'D15': '',
  'D16': ''},
 {'District': 'D2',
  'D1': '1',
  'D2': '1',
  'D3': '1',
  'D4': '',
  'D5': '1',
  'D6': '1',
  'D7': '',
  'D8': '',
  'D9': '',
  'D10': '',
  'D11': '',
  'D12': '',
  'D13': '',
  'D14': '',
  'D15': '',
  'D16': ''},
 {'District': 'D3',
  'D1': '',
  'D2': '1',
  'D3': '1',
  'D4': '',
  'D5': '',
  'D6': '1',
  'D7': '1',
  'D8': '',
  'D9': '',
  'D10': '',
  'D11': '',
  'D12': '',
  'D13': '',
  'D14': '',
  'D15': '',
  'D16': ''},
 {'District': 'D4',
  'D1': '1',
  'D2': '',
  'D3': '',


In [5]:
print(headers)

['District', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'D16']


In [6]:
Districts = headers[:]
Districts.remove('District')

In [7]:
print(Districts)

['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'D16']


In [8]:
Cost = {}
for i in Districts:
    Cost[i] = float(table[0][i])

In [9]:
print(Cost)

{'D1': 600.0, 'D2': 680.0, 'D3': 560.0, 'D4': 880.0, 'D5': 670.0, 'D6': 740.0, 'D7': 590.0, 'D8': 900.0, 'D9': 920.0, 'D10': 460.0, 'D11': 910.0, 'D12': 800.0, 'D13': 720.0, 'D14': 500.0, 'D15': 640.0, 'D16': 400.0}


In [10]:
for i in range(1,len(table)):
    print(i)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16


In [16]:
table[3]['District']

'D3'

In [27]:
# create a set of neighbors for each district
Neighbors = {}
# loop over the rows` of the adjacency matrix
for i in range(1,len(table)):
    # define empty set
    tmpSet = set()
    # loop over all districts (headers)
    for j in Districts:
        # if table == 1 : add the district to the set
        if table[i][j] == '1':
            tmpSet.add(j)
    # assign the set to the district corresponding to row i
    D = table[i]['District']
    Neighbors[D] = tmpSet

In [28]:
print(Neighbors)

{'D1': {'D2', 'D1', 'D4', 'D5'}, 'D2': {'D2', 'D1', 'D3', 'D6', 'D5'}, 'D3': {'D2', 'D3', 'D6', 'D7'}, 'D4': {'D1', 'D4', 'D11', 'D8', 'D5', 'D10'}, 'D5': {'D2', 'D1', 'D4', 'D6', 'D8', 'D5'}, 'D6': {'D2', 'D9', 'D3', 'D7', 'D6', 'D8', 'D5'}, 'D7': {'D9', 'D3', 'D7', 'D13', 'D6'}, 'D8': {'D9', 'D4', 'D11', 'D6', 'D8', 'D5', 'D12'}, 'D9': {'D9', 'D7', 'D13', 'D6', 'D8', 'D12'}, 'D10': {'D10', 'D4', 'D14', 'D11'}, 'D11': {'D12', 'D14', 'D4', 'D11', 'D8', 'D10'}, 'D12': {'D9', 'D11', 'D13', 'D15', 'D8', 'D12'}, 'D13': {'D16', 'D9', 'D7', 'D13', 'D15', 'D12'}, 'D14': {'D10', 'D15', 'D14', 'D11'}, 'D15': {'D16', 'D14', 'D13', 'D15', 'D12'}, 'D16': {'D15', 'D16', 'D13'}}


In [29]:
from docplex.mp.model import Model
mdl = Model()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [30]:
# variables
select = mdl.binary_var_dict(Districts, name='select')

In [31]:
# objective
mdl.minimize(mdl.sum(Cost[i]*select[i] for i in Districts))

In [32]:
# constraints: cover each district
CoverDistrict = {}
for j in Districts:
    CoverDistrict[j] = mdl.add_constraint(mdl.sum(select[i] for i in Neighbors[j]) >= 1)

In [33]:
print(CoverDistrict['D1'])

select_D1+select_D2+select_D4+select_D5 >= 1


In [40]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.0213079,status='integer optimal solution')

In [41]:
mdl.print_solution()

objective: 2260.000
status: OPTIMAL_SOLUTION(2)
  select_D4=1
  select_D6=1
  select_D15=1


In [42]:
# inspect how often each district is covered
for j in Districts:
    print("District " + j + ": " + str(CoverDistrict[j].lhs.solution_value))

District D1: 1.0
District D2: 1.0
District D3: 1.0
District D4: 1.0
District D5: 2.0
District D6: 1.0
District D7: 1.0
District D8: 2.0
District D9: 1.0
District D10: 1.0
District D11: 1.0
District D12: 1.0
District D13: 1.0
District D14: 1.0
District D15: 1.0
District D16: 1.0
